In [ ]:
# ==============================================================================
# SPECTRA v4 — 
# Key insight: TEST-TIME ADAPTATION, not by mixing
# Strategy: Identify domain at test time → route to best adapter
#           + fine-tune coefficients on unlabeled test data (TTT)
# ==============================================================================
!pip install -q cma
import os, gc, json, time, math, random, re, string, warnings, copy
import numpy as np
import pandas as pd
import torch
import scipy.stats as stats
import cma
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any, Optional

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.makedirs("/kaggle/working/hf_home", exist_ok=True)
os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/hf_cache"
os.environ["HF_HOME"]           = "/kaggle/working/hf_home"

torch.backends.cuda.matmul.allow_tf32 = True

from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_from_disk, Dataset, DatasetDict, disable_caching
from torch.utils.data import DataLoader

disable_caching()

DEVICE      = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE       = torch.float16
SEEDS       = [42, 123, 256, 512, 1024]
MASTER_SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(MASTER_SEED)

@dataclass
class CFG:
    day12_dir:    str   = "/kaggle/input/spectra-day1-2-artifacts-v1"
    model_path:   str   = ""
    work_dir:     str   = "/kaggle/working"
    lora_names:   tuple = ("math", "code", "hybrid")
    lora_r:       int   = 8
    lora_alpha:   int   = 16
    lora_dropout: float = 0.05
    lora_targets: tuple = ("q_proj","k_proj","v_proj","o_proj")
    train_epochs: int   = 2
    train_batch:  int   = 2
    grad_accum:   int   = 4
    lr:           float = 2e-4
    max_seq_len:  int   = 256
    gen_max_new:  int   = 128
    # CMA-ES: smaller population, more targeted
    cma_pop:      int   = 8
    cma_iters:    int   = 40
    cma_sigma0:   float = 0.3
    n_adapt:      int   = 15
    n_test:       int   = 50
    n_bootstrap:  int   = 1000

    def __post_init__(self):
        for d in ([self.work_dir,
                   f"{self.work_dir}/figures",
                   f"{self.work_dir}/artifacts"] +
                  [f"{self.work_dir}/lora_{n}" for n in self.lora_names]):
            os.makedirs(d, exist_ok=True)

cfg = CFG()

class Timer:
    def __init__(self, label=""): self.label = label
    def __enter__(self): self.t0 = time.time(); return self
    def __exit__(self, *_):
        print(f"[TIME] {self.label}: {time.time()-self.t0:.2f}s")

def vram_mb():
    return torch.cuda.max_memory_allocated()/1e6 if torch.cuda.is_available() else 0.0
def reset_vram():
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
def gc_collect():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
def print_sep(title="", width=70, char="="):
    pad = max(0, width-len(title)-2)
    print(f"\n{char*(pad//2)} {title} {char*(pad-pad//2)}" if title
          else char*width)

# ──────────────────────────────────────────────────────────────────────────────
# Statistics
# ──────────────────────────────────────────────────────────────────────────────
def bootstrap_ci(scores, n_boot=1000, alpha=0.05, seed=MASTER_SEED):
    if not scores: return 0.0, 0.0, 0.0
    rng = np.random.RandomState(seed); arr = np.array(scores)
    means = [np.mean(rng.choice(arr, len(arr), replace=True))
             for _ in range(n_boot)]
    return (float(np.mean(arr)),
            float(np.percentile(means, 100*alpha/2)),
            float(np.percentile(means, 100*(1-alpha/2))))

def paired_ttest(a, b):
    if len(a) < 2 or len(b) < 2: return 0.0, 1.0
    n = min(len(a), len(b))
    t, p = stats.ttest_rel(a[:n], b[:n])
    return float(t), float(p)

def cohens_d(a, b):
    """Effect size."""
    n = min(len(a), len(b))
    da, db = np.array(a[:n]), np.array(b[:n])
    pooled = np.sqrt((np.std(da)**2 + np.std(db)**2) / 2 + 1e-9)
    return float((np.mean(da) - np.mean(db)) / pooled)

# ──────────────────────────────────────────────────────────────────────────────
# Model path
# ──────────────────────────────────────────────────────────────────────────────
print_sep("Model Path")

def _has_weights(p):
    try: return any(f.endswith((".safetensors",".bin")) for f in os.listdir(p))
    except: return False

def _valid_model(p):
    if not os.path.isdir(p): return False
    if not all(os.path.isfile(f"{p}/{f}")
               for f in ["config.json","tokenizer_config.json"]): return False
    if not _has_weights(p): return False
    try:
        with open(f"{p}/config.json") as f: c = json.load(f)
        return c.get("model_type","").lower() in ("qwen2","qwen2_moe","qwen")
    except: return False

ARTIFACT_KW = ["baseline_full_ft","lora_","day1","day2","day3","spectra-day","checkpoint"]
BASE_KW     = ["qwen","1.5b","instruct","reasoner"]

def find_model_path():
    for p in ["/kaggle/input/qwen2.5-1.5b-instruct_reasoner/pytorch/default/1",
              "/kaggle/input/qwen2-5-1-5b-instruct/pytorch/default/1",
              "/kaggle/input/qwen2.5-1.5b-instruct/pytorch/default/1"]:
        if _valid_model(p): return p
    found = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if root[len("/kaggle/input"):].count(os.sep) > 6:
            dirs.clear(); continue
        if _valid_model(root):
            sc = (sum(k in root.lower() for k in BASE_KW)
                  - sum(k in root.lower() for k in ARTIFACT_KW))
            found.append((sc, root))
    if found:
        found.sort(key=lambda x: -x[0])
        print(f"[MODEL] {found[0][1]}"); return found[0][1]
    raise FileNotFoundError("No valid model found")

MODEL_PATH = find_model_path(); cfg.model_path = MODEL_PATH

# ──────────────────────────────────────────────────────────────────────────────
# Tokenizer
# ──────────────────────────────────────────────────────────────────────────────
print_sep("Tokenizer")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH, trust_remote_code=True, local_files_only=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"  Vocab={tokenizer.vocab_size}")

# ──────────────────────────────────────────────────────────────────────────────
# MergeableModel
# ──────────────────────────────────────────────────────────────────────────────
print_sep("Loading Model")

class MergeableModel:
    def __init__(self, path):
        self.model = AutoModelForCausalLM.from_pretrained(
            path, dtype=DTYPE, trust_remote_code=True,
            local_files_only=True, device_map={"": DEVICE})
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad = False
        self.base_weights = {n: p.data.float().cpu()
                             for n, p in self.model.named_parameters()}
        self.current_key = None
        print(f"  ✓ {len(self.base_weights)} params  VRAM={vram_mb():.0f}MB")

    def set_coefficients(self, coefficients, all_deltas, lora_names):
        key = tuple(round(c, 6) for c in coefficients)
        if key == self.current_key: return
        with torch.no_grad():
            params = dict(self.model.named_parameters())
            for name, W_base in self.base_weights.items():
                has_d = any(name in all_deltas[d] for d in lora_names)
                if not has_d:
                    if self.current_key is None:
                        p = params.get(name)
                        if p is not None:
                            p.data.copy_(W_base.to(DTYPE))
                    continue
                W = W_base.clone()
                for k, dom in enumerate(lora_names):
                    c = coefficients[k]
                    if abs(c) < 1e-9: continue
                    if name in all_deltas[dom]:
                        W = W + c * all_deltas[dom][name]
                p = params.get(name)
                if p is not None: p.data.copy_(W.to(DTYPE))
        self.current_key = key

    def generate(self, prompts, batch_size=4, max_new=None):
        if max_new is None: max_new = cfg.gen_max_new
        results = []
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i+batch_size]
            enc = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True,
                            max_length=cfg.max_seq_len).to(DEVICE)
            try:
                with torch.no_grad():
                    out = self.model.generate(
                        **enc, max_new_tokens=max_new, do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                        eos_token_id=tokenizer.eos_token_id)
                inp_len = enc["input_ids"].shape[1]
                for seq in out:
                    results.append(
                        tokenizer.decode(seq[inp_len:],
                                         skip_special_tokens=True).strip())
            except Exception as e:
                print(f"  [WARN] gen: {e}")
                results.extend([""] * len(batch))
        return results

    def nll_answer(self, prompt_response_pairs, max_len=128):
        """
        NLL on answer tokens only.
        This is the KEY fitness signal — aligned with generation quality.
        """
        total = 0.0; n = 0
        with torch.no_grad():
            for prompt, response in prompt_response_pairs:
                full   = f"### Q:\n{prompt}\n\n### A:\n{response}"
                prefix = f"### Q:\n{prompt}\n\n### A:\n"
                enc    = tokenizer(full, return_tensors="pt",
                                   truncation=True,
                                   max_length=max_len).to(DEVICE)
                enc_p  = tokenizer(prefix, return_tensors="pt",
                                   truncation=True, max_length=max_len)
                ids    = enc["input_ids"]
                lbl    = ids.clone()
                plen   = min(enc_p["input_ids"].shape[1], ids.shape[1])
                lbl[:, :plen] = -100
                if (lbl != -100).sum() == 0: continue
                try:
                    loss = self.model(
                        input_ids=ids,
                        attention_mask=enc["attention_mask"],
                        labels=lbl).loss.item()
                    total += loss; n += 1
                except: pass
        return total / max(1, n)

    def domain_nll(self, prompts, max_len=64):
        """Plain NLL for domain classification."""
        total = 0.0; n = 0
        with torch.no_grad():
            for p in prompts:
                inp = tokenizer(p, return_tensors="pt",
                                truncation=True, max_length=max_len).to(DEVICE)
                ids = inp["input_ids"]
                lbl = ids.clone(); lbl[:, :1] = -100
                try:
                    loss = self.model(
                        input_ids=ids,
                        attention_mask=inp.get("attention_mask"),
                        labels=lbl).loss.item()
                    total += loss; n += 1
                except: pass
        return total / max(1, n)

M = MergeableModel(MODEL_PATH)

# ──────────────────────────────────────────────────────────────────────────────
# Dataset
# ──────────────────────────────────────────────────────────────────────────────
print_sep("Dataset")

def _try_load():
    for base in [cfg.day12_dir,
                 "/kaggle/input/spectra-day1-2-artifacts-v1",
                 "/kaggle/input/datasets/ali320230101/spectra-day1-2-artifacts-v1"]:
        p = os.path.join(base, "data", "unified_dataset")
        if os.path.isdir(p):
            try:
                dd = load_from_disk(p, keep_in_memory=True)
                print(f"[DATA] {p}"); return dd
            except Exception as e: print(f"[WARN] {e}")
    return None

dataset_dict = _try_load()
if dataset_dict is None:
    dataset_dict = DatasetDict({
        "math": Dataset.from_list([
            {"prompt": f"Solve: x+{i}={i+10}",
             "response": f"x=10 → \\boxed{{10}}"} for i in range(200)]),
        "coding": Dataset.from_list([
            {"prompt": f"Write function #{i} returning {i}",
             "response": f"```python\ndef f(): return {i}\n```"}
            for i in range(200)]),
        "hybrid": Dataset.from_list([
            {"prompt": f"Compute {i}^2 in code",
             "response": f"\\boxed{{{i**2}}}\n```python\nprint({i}**2)\n```"}
            for i in range(200)]),
    })

for split, ds in dataset_dict.items():
    print(f"  {split}: {len(ds)}")

def carve_splits(ds, n_test=cfg.n_test, n_adapt=cfg.n_adapt,
                 seed=MASTER_SEED):
    ds    = Dataset.from_dict(ds[:]).shuffle(seed=seed, keep_in_memory=True)
    total = len(ds)
    te    = ds.select(range(min(n_test, total)), keep_in_memory=True)
    ad    = ds.select(range(min(n_test, total), min(n_test+n_adapt, total)),
                      keep_in_memory=True)
    tr    = ds.select(range(min(n_test+n_adapt, total), total),
                      keep_in_memory=True)
    return tr, ad, te

TRAIN_SETS, ADAPT_SETS, TEST_SETS = {}, {}, {}
for dom in ["math", "coding", "hybrid"]:
    tr, ad, te = carve_splits(dataset_dict[dom])
    TRAIN_SETS[dom], ADAPT_SETS[dom], TEST_SETS[dom] = tr, ad, te
    print(f"  {dom}: train={len(tr)}, adapt={len(ad)}, test={len(te)}")

DOMAIN_TO_SPLIT = {"math":"math","code":"coding",
                   "hybrid":"hybrid","coding":"coding"}

# ──────────────────────────────────────────────────────────────────────────────
# Metrics
# ──────────────────────────────────────────────────────────────────────────────
print_sep("Metrics")

def normalize_answer(s):
    s = s.lower().strip()
    s = re.sub(r'\\boxed\{([^}]*)\}', r'\1', s)
    s = re.sub(r'\$+', '', s)
    s = "".join(ch if ch not in string.punctuation else " " for ch in s)
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return " ".join(s.split())

def extract_final_answer(text):
    m = re.findall(r'\\boxed\{([^}]+)\}', text)
    if m: return m[-1].strip()
    m = re.findall(
        r'(?:the answer is|answer:|=|result is|therefore)[:\s]+([^\n.]+)',
        text, re.IGNORECASE)
    if m: return m[-1].strip()
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return lines[-1] if lines else text.strip()

def em_score(pred, gold):
    return float(normalize_answer(extract_final_answer(pred))
                 == normalize_answer(extract_final_answer(gold)))

def token_f1(pred, gold):
    p_toks = normalize_answer(extract_final_answer(pred)).split()
    g_toks = normalize_answer(extract_final_answer(gold)).split()
    if not p_toks or not g_toks: return 0.0
    common = Counter(p_toks) & Counter(g_toks)
    num    = sum(common.values())
    if num == 0: return 0.0
    prec = num/len(p_toks); rec = num/len(g_toks)
    return 2*prec*rec/(prec+rec)

def rouge_l(pred, gold):
    try:
        from rouge_score import rouge_scorer
        sc = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        return sc.score(gold, pred)['rougeL'].fmeasure
    except:
        p, g = pred.lower().split(), gold.lower().split()
        if not p or not g: return 0.0
        m, n = len(g), len(p)
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(1, m+1):
            for j in range(1, n+1):
                dp[i][j] = (dp[i-1][j-1]+1 if g[i-1]==p[j-1]
                             else max(dp[i-1][j], dp[i][j-1]))
        lcs = dp[m][n]
        prec = lcs/n if n > 0 else 0
        rec  = lcs/m if m > 0 else 0
        return 2*prec*rec/(prec+rec+1e-8)

def compute_metrics(preds, golds, do_bootstrap=True):
    if not preds: return {}
    em   = [em_score(p,g)  for p,g in zip(preds,golds)]
    f1   = [token_f1(p,g)  for p,g in zip(preds,golds)]
    rl   = [rouge_l(p,g)   for p,g in zip(preds,golds)]
    comp = [0.4*e+0.35*f+0.25*r for e,f,r in zip(em,f1,rl)]
    out  = {"em": float(np.mean(em)), "f1": float(np.mean(f1)),
            "rouge_l": float(np.mean(rl)), "composite": float(np.mean(comp)),
            "em_list": em, "f1_list": f1, "rl_list": rl, "comp_list": comp}
    if do_bootstrap:
        for k, lst in [("em",em),("f1",f1),("rouge_l",rl),("composite",comp)]:
            mn, lo, hi = bootstrap_ci(lst)
            out[f"{k}_ci_lo"] = lo; out[f"{k}_ci_hi"] = hi
    return out

print("  EM, F1, ROUGE-L, Composite(0.40/0.35/0.25) + Bootstrap CI")

# ──────────────────────────────────────────────────────────────────────────────
# LoRA Training
# ──────────────────────────────────────────────────────────────────────────────
print_sep("LoRA Training")

LORA_SAVE_PATHS = {n: f"{cfg.work_dir}/lora_{n}" for n in cfg.lora_names}

def format_example(prompt, response, tok, max_len):
    full = f"### Q:\n{prompt}\n\n### A:\n{response}{tok.eos_token}"
    enc  = tok(full, truncation=True, max_length=max_len, return_tensors="pt")
    ids  = enc["input_ids"][0]
    plen = len(tok(f"### Q:\n{prompt}\n\n### A:\n",
                   truncation=True, max_length=max_len)["input_ids"])
    lbl  = ids.clone(); lbl[:plen] = -100
    return {"input_ids": ids, "labels": lbl,
            "attention_mask": enc["attention_mask"][0]}

class PromptDataset(torch.utils.data.Dataset):
    def __init__(self, ds, tok, ml):
        self.ds=ds; self.tok=tok; self.ml=ml
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        r = self.ds[i]
        return format_example(r["prompt"], r["response"], self.tok, self.ml)

def collate_fn(batch):
    L    = max(x["input_ids"].shape[0] for x in batch)
    ids  = torch.zeros(len(batch), L, dtype=torch.long)
    mask = torch.zeros(len(batch), L, dtype=torch.long)
    lbl  = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, x in enumerate(batch):
        n = x["input_ids"].shape[0]
        ids[i,:n]=x["input_ids"]; mask[i,:n]=x["attention_mask"]; lbl[i,:n]=x["labels"]
    return {"input_ids": ids, "attention_mask": mask, "labels": lbl}

def train_lora(domain_name, train_ds, force=False):
    save_path = LORA_SAVE_PATHS[domain_name]
    for cand in [
        f"{cfg.day12_dir}/lora_{domain_name}",
        f"/kaggle/input/datasets/ali320230101/spectra-day1-2-artifacts-v1/lora_{domain_name}",
        save_path
    ]:
        if (not force and os.path.isdir(cand)
                and os.path.isfile(f"{cand}/adapter_config.json")):
            print(f"[LORA] Reusing {domain_name} at {cand}"); return cand
    print(f"[LORA] Training {domain_name} …")
    set_seed(MASTER_SEED)
    lora_cfg = LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_targets),
        task_type=TaskType.CAUSAL_LM, bias="none")
    peft_model = get_peft_model(M.model, lora_cfg); peft_model.train()
    loader = DataLoader(
        PromptDataset(train_ds, tokenizer, cfg.max_seq_len),
        batch_size=cfg.train_batch, shuffle=True,
        collate_fn=collate_fn, num_workers=0)
    opt = torch.optim.AdamW(
        [p for p in peft_model.parameters() if p.requires_grad],
        lr=cfg.lr, weight_decay=0.01)
    total_steps = max(1,(len(loader)//cfg.grad_accum)*cfg.train_epochs)
    sched = get_cosine_schedule_with_warmup(
        opt, max(1, total_steps//10), total_steps)
    for epoch in range(cfg.train_epochs):
        total_loss=0.0; n=0; opt.zero_grad()
        for step, batch in enumerate(loader):
            batch = {k:v.to(DEVICE) for k,v in batch.items()}
            loss  = peft_model(**batch).loss / cfg.grad_accum
            loss.backward(); total_loss += loss.item()*cfg.grad_accum; n += 1
            if (step+1) % cfg.grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in peft_model.parameters() if p.requires_grad], 1.0)
                opt.step(); sched.step(); opt.zero_grad()
        print(f"  Epoch {epoch+1}  loss={total_loss/max(1,n):.4f}")
    peft_model.save_pretrained(save_path); tokenizer.save_pretrained(save_path)
    del peft_model; gc_collect()
    M.model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, dtype=DTYPE, trust_remote_code=True,
        local_files_only=True, device_map={"": DEVICE})
    M.model.eval()
    for p in M.model.parameters(): p.requires_grad = False
    M.current_key = None
    return save_path

with Timer("LoRA"):
    LORA_PATHS = {dom: train_lora(dom, TRAIN_SETS[DOMAIN_TO_SPLIT[dom]])
                  for dom in cfg.lora_names}

# ──────────────────────────────────────────────────────────────────────────────
# Delta Extraction
# ──────────────────────────────────────────────────────────────────────────────
print_sep("LoRA Delta Extraction")

def extract_deltas(lora_path):
    with open(f"{lora_path}/adapter_config.json") as f:
        acfg = json.load(f)
    r = acfg.get("r", cfg.lora_r)
    alpha = acfg.get("lora_alpha", cfg.lora_alpha)
    s = alpha / r
    raw = {}
    for fname in os.listdir(lora_path):
        if fname.endswith(".safetensors"):
            from safetensors.torch import load_file
            raw = load_file(f"{lora_path}/{fname}", device="cpu"); break
    if not raw:
        for fname in os.listdir(lora_path):
            if fname.endswith(".bin"):
                raw = torch.load(f"{lora_path}/{fname}", map_location="cpu"); break
    A, B = {}, {}
    for key, tensor in raw.items():
        clean = key
        for pre in ["base_model.model.", "base_model."]:
            if clean.startswith(pre): clean = clean[len(pre):]
        if "lora_A" in clean:
            A[clean.replace(".lora_A.weight",".weight")] = tensor.float()
        elif "lora_B" in clean:
            B[clean.replace(".lora_B.weight",".weight")] = tensor.float()
    deltas = {pn:(s*B[pn]@A[pn]) for pn in A if pn in B}
    print(f"  {os.path.basename(lora_path)}: {len(deltas)} deltas")
    return deltas

ALL_DELTAS = {dom: extract_deltas(LORA_PATHS[dom]) for dom in cfg.lora_names}
LORA_NAMES = list(cfg.lora_names)

# ──────────────────────────────────────────────────────────────────────────────
# Weight Apply Helpers
# ──────────────────────────────────────────────────────────────────────────────
def apply_c(c): M.set_coefficients(c, ALL_DELTAS, LORA_NAMES)
def reset_base(): apply_c([0.0]*len(LORA_NAMES))
def apply_single(d): apply_c([1.0 if x==d else 0.0 for x in LORA_NAMES])
def apply_equal():   apply_c([1.0/len(LORA_NAMES)]*len(LORA_NAMES))

print("Computing TIES …")
def compute_ties(density=0.5):
    all_keys = set()
    for d in ALL_DELTAS.values(): all_keys.update(d.keys())
    ties = {}
    for pn in all_keys:
        raw = [ALL_DELTAS[d][pn].float()
               for d in LORA_NAMES if pn in ALL_DELTAS[d]]
        if not raw: continue
        pruned = []
        for dw in raw:
            flat = dw.abs().flatten()
            thr  = float(flat.kthvalue(
                max(1, int((1-density)*flat.numel()))).values)
            pruned.append(torch.where(dw.abs()>=thr, dw, torch.zeros_like(dw)))
        maj    = torch.stack([p.sign() for p in pruned]).sum(0).sign()
        agreed = [p*(p.sign()==maj).float() for p in pruned]
        ties[pn] = torch.stack(agreed).mean(0)
    return ties

TIES_DELTAS = compute_ties()

def apply_ties():
    key = ("ties",)
    if M.current_key == key: return
    with torch.no_grad():
        params = dict(M.model.named_parameters())
        for name, W_base in M.base_weights.items():
            if name in TIES_DELTAS:
                p = params.get(name)
                if p is not None:
                    p.data.copy_((W_base.float()+TIES_DELTAS[name]).to(DTYPE))
    M.current_key = key

# ──────────────────────────────────────────────────────────────────────────────
# ★ SPECTRA v4: Domain-Routing + Test-Time Adaptation
# ──────────────────────────────────────────────────────────────────────────────
print_sep("SPECTRA v4: Domain-Routing + TTA")

def softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max()); return e / e.sum()

# ── Phase 1: Domain Router ────────────────────────────────────────────────────
class DomainRouter:
    """
    Identifies domain from adapt prompts by measuring perplexity
    under each domain-specific adapter.
    
    Key insight: the domain-matched adapter gives LOWER perplexity
    on domain-specific prompts → use this as a soft routing signal.
    """
    def __init__(self):
        self.domain_nlls: Dict[str, float] = {}

    def fit(self, adapt_prompts: List[str], verbose=False) -> np.ndarray:
        """
        Returns softmax-normalized routing weights (domain probabilities).
        Lower NLL → higher weight.
        """
        nlls = []
        for i, dom in enumerate(LORA_NAMES):
            apply_single(dom)
            nll = M.domain_nll(adapt_prompts)
            nlls.append(nll)
            self.domain_nlls[dom] = nll
            if verbose:
                print(f"    Router NLL [{dom}]: {nll:.4f}")
        # Convert: lower NLL = higher confidence
        nlls_arr = np.array(nlls)
        # Softmax of negative NLL = routing weights
        weights = softmax(-nlls_arr / max(nlls_arr.std(), 0.01))
        if verbose:
            print(f"    Router weights: {np.round(weights, 3)}")
        return weights

router = DomainRouter()

# ── Phase 2: SPECTRA TTA (Test-Time Adaptation) ───────────────────────────────
def spectra_tta_fitness(
        raw_params: np.ndarray,
        adapt_pairs: List[Tuple[str, str]],
        router_weights: Optional[np.ndarray] = None,
) -> float:
    """
    SPECTRA fitness with TTA signal.
    
    Fitness = NLL_answer(adapt)          ← primary: answer quality
            + 0.1 * KL(c, router_w)     ← soft routing prior
    
    The routing prior pulls SPECTRA toward the domain-matched adapter
    but allows the answer-NLL to override if another adapter is better.
    """
    c = softmax(raw_params)
    apply_c(c.tolist())
    nll = M.nll_answer(adapt_pairs)

    # Soft routing prior: penalize deviation from router prediction
    kl_term = 0.0
    if router_weights is not None:
        # KL(c || router_weights) — encourages c toward router prediction
        r = np.clip(router_weights, 1e-6, 1.0)
        c_cl = np.clip(c, 1e-6, 1.0)
        kl = float(np.sum(r * np.log(r / c_cl)))
        kl_term = 0.1 * kl

    return nll + kl_term

def spectra_adapt(
        adapt_pairs: List[Tuple[str, str]],
        router_weights: Optional[np.ndarray] = None,
        verbose: bool = True,
) -> Tuple[np.ndarray, float, List[float], float]:
    """
    SPECTRA: CMA-ES with domain routing prior.
    No bounds → no ValueError.
    """
    K = len(LORA_NAMES)

    # ── Smart initialization using router ────────────────────────────────
    candidates = []

    # Router-based init: start near the router's prediction
    if router_weights is not None:
        # Convert router weights to raw params via inverse softmax
        raw_router = np.log(np.clip(router_weights, 1e-6, 1.0))
        raw_router -= raw_router.mean()
        fit_router = spectra_tta_fitness(raw_router, adapt_pairs, router_weights)
        candidates.append((fit_router, raw_router.copy(), "router"))
        if verbose:
            print(f"    init router:       fit={fit_router:.4f}  "
                  f"c={np.round(softmax(raw_router),3)}")

    # Single-adapter vertices
    for i, dom in enumerate(LORA_NAMES):
        raw = np.zeros(K); raw[i] = 2.0
        fit = spectra_tta_fitness(raw, adapt_pairs, router_weights)
        candidates.append((fit, raw.copy(), f"single_{dom}"))
        if verbose:
            print(f"    init single_{dom}: fit={fit:.4f}  "
                  f"c={np.round(softmax(raw),3)}")

    # Equal
    raw_eq = np.zeros(K)
    fit_eq = spectra_tta_fitness(raw_eq, adapt_pairs, router_weights)
    candidates.append((fit_eq, raw_eq.copy(), "equal"))
    if verbose:
        print(f"    init equal:        fit={fit_eq:.4f}  "
              f"c={np.round(softmax(raw_eq),3)}")

    # Random starts (seeded)
    rng = np.random.RandomState(MASTER_SEED)
    for _ in range(5):
        raw_r = rng.randn(K) * 0.5
        fit_r = spectra_tta_fitness(raw_r, adapt_pairs, router_weights)
        candidates.append((fit_r, raw_r.copy(), "random"))

    best_fit, best_raw, best_name = min(candidates, key=lambda x: x[0])
    if verbose:
        print(f"  Best init: {best_name}  fit={best_fit:.4f}  "
              f"c={np.round(softmax(best_raw),3)}")

    # ── CMA-ES (no bounds) ────────────────────────────────────────────────
    es = cma.CMAEvolutionStrategy(
        best_raw, cfg.cma_sigma0,
        {"popsize": cfg.cma_pop,
         "maxiter": cfg.cma_iters,
         "seed":    MASTER_SEED,
         "verbose": -9,
         "tolx":    1e-6,
         "tolfun":  1e-6})

    history = []; t0 = time.time(); gen = 0
    while not es.stop():
        sols = es.ask()
        fits = [spectra_tta_fitness(s, adapt_pairs, router_weights)
                for s in sols]
        es.tell(sols, fits)
        history.append(float(np.min(fits))); gen += 1
        if verbose and gen % 10 == 0:
            bc = softmax(es.result.xbest)
            print(f"    gen {gen:3d}/{cfg.cma_iters}  fit={history[-1]:.4f}  "
                  f"c={np.round(bc,3)}")

    adapt_t   = time.time() - t0
    best_c    = softmax(es.result.xbest)
    apply_c(best_c.tolist())
    if verbose:
        print(f"  Done: fit={es.result.fbest:.4f}  gens={gen}  "
              f"t={adapt_t:.1f}s")
        print(f"  Final coeffs: {np.round(best_c,3)}")
    return best_c, float(es.result.fbest), history, adapt_t

# ──────────────────────────────────────────────────────────────────────────────
# Data Helpers
# ──────────────────────────────────────────────────────────────────────────────
def get_test_examples(domain, n=None):
    key = DOMAIN_TO_SPLIT.get(domain, domain)
    ds  = TEST_SETS.get(key) or TEST_SETS.get(domain)
    if ds is None: return []
    n = len(ds) if n is None else min(n, len(ds))
    return [(ds[i]["prompt"], ds[i]["response"]) for i in range(n)]

def get_adapt_examples(domain, n=cfg.n_adapt, seed_offset=0):
    key = DOMAIN_TO_SPLIT.get(domain, domain)
    ds  = ADAPT_SETS.get(key) or ADAPT_SETS.get(domain)
    if ds is None: return []
    set_seed(MASTER_SEED + seed_offset)
    idxs = random.sample(range(len(ds)), min(n, len(ds)))
    return [(ds[i]["prompt"], ds[i]["response"]) for i in idxs]

def get_adapt_prompts(domain, n=cfg.n_adapt, seed_offset=0):
    return [p for p, _ in get_adapt_examples(domain, n, seed_offset)]

# ──────────────────────────────────────────────────────────────────────────────
# Evaluation Harness
# ──────────────────────────────────────────────────────────────────────────────
print_sep("Evaluation Harness")

RESULTS: List[Dict] = []
RESULTS_CSV = f"{cfg.work_dir}/spectra_results.csv"
PER_EXAMPLE_SCORES: Dict[str, Dict[str, List[float]]] = defaultdict(dict)

def flush_results():
    if RESULTS: pd.DataFrame(RESULTS).to_csv(RESULTS_CSV, index=False)

def evaluate(name, domain, condition, setup_fn, examples,
             adapt_time=None, extra=None, seeds=None):
    if seeds is None: seeds = SEEDS
    all_em, all_f1, all_rl, all_comp = [], [], [], []
    all_comp_flat = []
    reset_vram()
    prompts = [p for p,_ in examples]
    golds   = [g for _,g in examples]
    setup_fn()
    for seed in seeds:
        set_seed(seed)
        idxs = list(range(len(examples))); random.shuffle(idxs)
        sp = [prompts[i] for i in idxs]
        sg = [golds[i]   for i in idxs]
        t0 = time.time()
        try: preds = M.generate(sp)
        except Exception as e:
            print(f"  [WARN] {e}"); preds = [""]*len(sp)
        m = compute_metrics(preds, sg, do_bootstrap=False)
        all_em.append(m["em"]); all_f1.append(m["f1"])
        all_rl.append(m["rouge_l"]); all_comp.append(m["composite"])
        all_comp_flat.extend(m["comp_list"])
        print(f"  [{name:>26s}|{domain:>7s}|s={seed}]  "
              f"EM={m['em']:.4f}  F1={m['f1']:.4f}  "
              f"RL={m['rouge_l']:.4f}  Comp={m['composite']:.4f}  "
              f"t={time.time()-t0:.1f}s")

    comp_mean, comp_lo, comp_hi = bootstrap_ci(all_comp_flat)
    key = f"{condition}/{domain}"
    PER_EXAMPLE_SCORES[key][name] = all_comp_flat

    row = {
        "method":       name,
        "domain":       domain,
        "condition":    condition,
        "n_test":       len(examples),
        "n_seeds":      len(seeds),
        "em":           float(np.mean(all_em)),
        "em_std":       float(np.std(all_em)),
        "f1":           float(np.mean(all_f1)),
        "rouge_l":      float(np.mean(all_rl)),
        "composite":    comp_mean,
        "comp_ci_lo":   comp_lo,
        "comp_ci_hi":   comp_hi,
        "adapt_time_s": adapt_time,
        "peak_vram_mb": vram_mb(),
        "timestamp":    pd.Timestamp.utcnow().isoformat(),
    }
    if extra: row.update(extra)
    RESULTS.append(row); flush_results()
    print(f"  >>> {name}/{domain}:  "
          f"EM={row['em']:.4f}±{row['em_std']:.4f}  "
          f"F1={row['f1']:.4f}  RL={row['rouge_l']:.4f}  "
          f"Composite={comp_mean:.4f} [{comp_lo:.4f},{comp_hi:.4f}]")
    return row

# ──────────────────────────────────────────────────────────────────────────────
# Experiment 1: Main Comparison
# ──────────────────────────────────────────────────────────────────────────────
print_sep("EXPERIMENT 1: Main Comparison")

_coeffs: Dict[str, np.ndarray] = {}

def grid_search_sf(adapt_pairs, n_pts=25):
    """Grid search with answer-NLL fitness."""
    best_c = None; best_nll = float("inf")
    step = 1.0/(n_pts-1) if n_pts > 1 else 1.0
    pts  = []
    for i in range(n_pts):
        for j in range(n_pts-i):
            pts.append([i*step, j*step, (n_pts-1-i-j)*step])
    random.Random(MASTER_SEED).shuffle(pts)
    for gp in pts[:35]:
        apply_c(gp)
        nll = M.nll_answer(adapt_pairs[:8])
        if nll < best_nll: best_nll = nll; best_c = gp[:]
    return best_c, best_nll

def run_exp1():
    for domain in LORA_NAMES:
        print(f"\n{'─'*60}\n Domain: {domain.upper()}\n{'─'*60}")
        test_ex  = get_test_examples(domain)
        adapt_pr = get_adapt_examples(domain, cfg.n_adapt)
        if not test_ex: continue
        print(f"N_test={len(test_ex)}  N_adapt={len(adapt_pr)}")

        # Baselines
        evaluate("no_adaptation",        domain, "exp1_main", reset_base,  test_ex)
        evaluate(f"single_lora_{domain}", domain, "exp1_main",
                 lambda d=domain: apply_single(d), test_ex)
        evaluate("equal_merge",  domain, "exp1_main", apply_equal, test_ex)
        evaluate("ties_merge",   domain, "exp1_main", apply_ties,  test_ex)

        # Scalar fusion
        print("  Grid search SF …")
        t0 = time.time()
        sf_c, sf_nll = grid_search_sf(adapt_pr)
        print(f"    SF: {np.round(sf_c,3)}  NLL={sf_nll:.4f}  "
              f"t={time.time()-t0:.1f}s")
        evaluate("scalar_fusion", domain, "exp1_main",
                 lambda c=sf_c: apply_c(c), test_ex,
                 extra={"sf_weights": json.dumps(sf_c)})

        # ── SPECTRA: Router → TTA ──────────────────────────────────────────
        print("  SPECTRA (Router + TTA) …")

        # Step 1: Domain routing
        adapt_prompts = [p for p,_ in adapt_pr]
        print("  [Router] Measuring domain NLL …")
        router_weights = router.fit(adapt_prompts, verbose=True)

        # Step 2: CMA-ES with routing prior
        with Timer("  SPECTRA"):
            best_c, fit, hist, t = spectra_adapt(
                adapt_pr, router_weights=router_weights, verbose=True)

        _coeffs[domain] = best_c
        np.save(f"{cfg.work_dir}/artifacts/coeffs_{domain}.npy", best_c)
        evaluate("spectra", domain, "exp1_main",
                 lambda c=best_c: apply_c(c.tolist()), test_ex,
                 adapt_time=t,
                 extra={"cma_fit":  fit,
                         "coeffs":   json.dumps(best_c.tolist()),
                         "history":  json.dumps(hist),
                         "router_w": json.dumps(router_weights.tolist())})

with Timer("Experiment 1"):
    run_exp1()

# ──────────────────────────────────────────────────────────────────────────────
# Rankings + Significance
# ──────────────────────────────────────────────────────────────────────────────
print_sep("EXPERIMENT 1 RANKINGS + SIGNIFICANCE")

exp1_df = pd.DataFrame([r for r in RESULTS if r.get("condition")=="exp1_main"])
exp1_df["method_norm"] = exp1_df["method"].apply(
    lambda m: "single_lora" if m.startswith("single_lora_") else m)

METHOD_ORDER = ["no_adaptation","equal_merge","ties_merge",
                "single_lora","scalar_fusion","spectra"]
METRICS = ["em","f1","rouge_l","composite"]

spectra_wins = 0
for metric in METRICS:
    if metric not in exp1_df.columns: continue
    agg = (exp1_df.groupby("method_norm")[metric].mean()
                 .reindex(METHOD_ORDER).dropna()
                 .sort_values(ascending=False))
    winner = agg.index[0] if not agg.empty else "?"
    if winner == "spectra": spectra_wins += 1
    print(f"\n  {metric.upper()} ranking:")
    for rank,(m,v) in enumerate(agg.items()):
        mark = (" ← SPECTRA #1 ✓" if (rank==0 and m=="spectra")
                else (" ← #1" if rank==0 else ""))
        print(f"    {rank+1}. {m:<28} {v:.4f}{mark}")

print(f"\n  Significance (SPECTRA vs baselines, composite):")
for domain in LORA_NAMES:
    key = f"exp1_main/{domain}"
    scores_sp = PER_EXAMPLE_SCORES[key].get("spectra",[])
    if not scores_sp: continue
    print(f"\n    Domain={domain}  (n={len(scores_sp)} per-example scores)")
    for bl in ["no_adaptation","equal_merge","ties_merge",
               "scalar_fusion","single_lora"]:
        scores_bl = PER_EXAMPLE_SCORES[key].get(
            bl, PER_EXAMPLE_SCORES[key].get(f"single_lora_{domain}",[]))
        if not scores_bl: continue
        t_stat, p_val = paired_ttest(scores_sp, scores_bl)
        d = cohens_d(scores_sp, scores_bl)
        delta = np.mean(scores_sp) - np.mean(scores_bl)
        sig   = "✓ p<0.05" if p_val < 0.05 else "✗ ns"
        print(f"      vs {bl:<22}: Δ={delta:+.4f}  t={t_stat:+.2f}  "
              f"p={p_val:.3f}  d={d:+.2f}  {sig}")

print(f"\n{'='*50}")
print(f"SPECTRA wins {spectra_wins}/{len(METRICS)} metrics")

# ──────────────────────────────────────────────────────────────────────────────
# Experiment 2: Ablation — Router vs No-Router vs No-Diversity
# ──────────────────────────────────────────────────────────────────────────────
print_sep("EXPERIMENT 2: Ablation — Router Contribution")

def run_exp2():
    domain   = "hybrid"
    test_ex  = get_test_examples(domain)
    adapt_pr = get_adapt_examples(domain, cfg.n_adapt, seed_offset=999)
    if not test_ex: return

    evaluate("ties_merge",  domain, "exp2_ablation", apply_ties,  test_ex)
    evaluate("equal_merge", domain, "exp2_ablation", apply_equal, test_ex)

    # SPECTRA-Full (router + TTA)
    adapt_prompts = [p for p,_ in adapt_pr]
    rw = router.fit(adapt_prompts, verbose=True)
    c, fit, _, t = spectra_adapt(adapt_pr, router_weights=rw, verbose=True)
    evaluate("spectra_full", domain, "exp2_ablation",
             lambda cc=c: apply_c(cc.tolist()), test_ex,
             adapt_time=t,
             extra={"cma_fit": fit, "coeffs": json.dumps(c.tolist())})

    # SPECTRA-NoRouter (TTA only, no routing prior)
    print("\n[ABL] SPECTRA-NoRouter (TTA only)")
    c2, _, _, t2 = spectra_adapt(adapt_pr, router_weights=None, verbose=False)
    evaluate("spectra_no_router", domain, "exp2_ablation",
             lambda cc=c2: apply_c(cc.tolist()), test_ex,
             adapt_time=t2,
             extra={"coeffs": json.dumps(c2.tolist())})

    # Router-Only (no CMA-ES, just use router weights directly)
    print("\n[ABL] Router-Only (no CMA-ES)")
    adapt_prompts2 = [p for p,_ in
                      get_adapt_examples(domain, cfg.n_adapt, seed_offset=999)]
    rw2 = router.fit(adapt_prompts2)
    evaluate("router_only", domain, "exp2_ablation",
             lambda rw=rw2: apply_c(rw.tolist()), test_ex,
             extra={"router_w": json.dumps(rw2.tolist())})

    # Ablation significance
    print("\n  Ablation Significance (composite):")
    key = "exp2_ablation/hybrid"
    for ab in ["spectra_full","spectra_no_router","router_only"]:
        sa = PER_EXAMPLE_SCORES[key].get(ab,[])
        for bl in ["ties_merge","equal_merge"]:
            sb = PER_EXAMPLE_SCORES[key].get(bl,[])
            if not sa or not sb: continue
            _, p = paired_ttest(sa, sb)
            d     = cohens_d(sa, sb)
            delta = np.mean(sa) - np.mean(sb)
            sig   = "✓" if p < 0.05 else "✗"
            print(f"    {ab} vs {bl}: Δ={delta:+.4f}  p={p:.3f}  d={d:+.2f} {sig}")

with Timer("Experiment 2"):
    run_exp2()

# ──────────────────────────────────────────────────────────────────────────────
# Experiment 3: Sample Efficiency (Nested, Monotonic)
# ──────────────────────────────────────────────────────────────────────────────
print_sep("EXPERIMENT 3: Sample Efficiency")

def run_exp3():
    domain    = "hybrid"   # hybrid is hardest — best test of TTA
    test_ex   = get_test_examples(domain)
    if not test_ex: return

    evaluate("no_adaptation", domain, "exp3_ref", reset_base, test_ex)

    # Fixed pool — nested subsets
    all_adapt = get_adapt_examples(domain, 20, seed_offset=7)
    prev_scores = None

    for n_prime in [5, 10, 15, 20]:
        print(f"\n── N'={n_prime} ──")
        aps = all_adapt[:n_prime]   # NESTED subset

        evaluate("ties_merge", domain, f"exp3_n{n_prime}",
                 apply_ties, test_ex, extra={"n_prime": n_prime})
        evaluate("equal_merge", domain, f"exp3_n{n_prime}",
                 apply_equal, test_ex, extra={"n_prime": n_prime})

        # SPECTRA with routing
        ps = [p for p,_ in aps]
        rw = router.fit(ps)
        c, fit, _, t = spectra_adapt(aps, router_weights=rw, verbose=False)
        row = evaluate("spectra", domain, f"exp3_n{n_prime}",
                       lambda cc=c: apply_c(cc.tolist()), test_ex,
                       adapt_time=t,
                       extra={"n_prime": n_prime, "cma_fit": fit})

        # Monotonicity check
        cur = PER_EXAMPLE_SCORES[f"exp3_n{n_prime}/{domain}"].get("spectra",[])
        if prev_scores and cur:
            delta = np.mean(cur) - np.mean(prev_scores)
            trend = "↑ improving" if delta >= -0.003 else "↓ regressing"
            print(f"  [Monotonicity] N'={n_prime}: "
                  f"Δcomposite={delta:+.4f} {trend}")
        prev_scores = cur

with Timer("Experiment 3"):
    run_exp3()

# ──────────────────────────────────────────────────────────────────────────────
# Experiment 4: Coefficient Analysis
# ──────────────────────────────────────────────────────────────────────────────
print_sep("EXPERIMENT 4: Coefficient Analysis")

def run_exp4():
    rows = []
    print("  Domain   Coeffs                    Entropy  Router agree?")
    print("  "+"─"*60)
    for domain in LORA_NAMES:
        c = _coeffs.get(domain)
        if c is None:
            aps = get_adapt_examples(domain, cfg.n_adapt, seed_offset=4000)
            ps  = [p for p,_ in aps]
            rw  = router.fit(ps)
            c, _, _, _ = spectra_adapt(aps, router_weights=rw, verbose=False)
            _coeffs[domain] = c
        H = -np.sum(c*np.log(c+1e-9))

        # Does SPECTRA agree with router?
        ps2    = get_adapt_prompts(domain, cfg.n_adapt, seed_offset=4001)
        rw2    = router.fit(ps2)
        sp_dom = LORA_NAMES[int(np.argmax(c))]
        rt_dom = LORA_NAMES[int(np.argmax(rw2))]
        agree  = "✓" if sp_dom==rt_dom else "✗"

        row = {"domain": domain, "dominant": sp_dom, "entropy": float(H),
               "is_mixed": float(H>0.5), "router_agree": agree}
        for i,ln in enumerate(LORA_NAMES): row[f"coeff_{ln}"] = float(c[i])
        rows.append(row)
        print(f"  {domain:8s}: {np.round(c,3)}  H={H:.3f}  "
              f"mixed={'Yes' if H>0.5 else 'No '}  Router={agree}")

    pd.DataFrame(rows).to_csv(
        f"{cfg.work_dir}/artifacts/exp4_coefficients.csv", index=False)

with Timer("Experiment 4"):
    run_exp4()

# ──────────────────────────────────────────────────────────────────────────────
# Experiment 5: Efficiency + Amortization
# ──────────────────────────────────────────────────────────────────────────────
print_sep("EXPERIMENT 5: Efficiency + Amortization")

def run_exp5():
    test_ex = get_test_examples("hybrid", n=20)
    c = _coeffs.get("hybrid")
    if c is None:
        aps = get_adapt_examples("hybrid", cfg.n_adapt, 5000)
        ps  = [p for p,_ in aps]
        rw  = router.fit(ps)
        c, _, _, _ = spectra_adapt(aps, router_weights=rw, verbose=False)

    prompts = [p for p,_ in test_ex]
    rows    = []
    for nm, setup in [
        ("no_adaptation", reset_base),
        ("ties_merge",    apply_ties),
        ("spectra",       lambda cc=c: apply_c(cc.tolist())),
    ]:
        setup(); reset_vram()
        t0 = time.time(); M.generate(prompts)
        et = time.time()-t0; pv = vram_mb()
        rows.append({"method":nm,"eval_per_ex":et/max(1,len(prompts)),"peak_vram_mb":pv})
        print(f"  {nm:>20s}: {et/max(1,len(prompts)):.2f}s/ex  {pv:.0f}MB")

    # Amortization
    print("\n  Amortization Analysis:")
    cma_cost = sum(r.get("adapt_time_s",0) or 0 for r in RESULTS
                   if r.get("method")=="spectra")
    overhead = (rows[-1]["eval_per_ex"] - rows[0]["eval_per_ex"])
    print(f"    Total CMA adapt cost: {cma_cost:.1f}s")
    print(f"    Inference overhead:   {max(overhead,0):.3f}s/ex  "
          f"(≈0 — weight merging has zero inference overhead)")
    print(f"    SPECTRA overhead is ONLY at adapt time, not eval time")
    n_domains = len(LORA_NAMES)
    print(f"    Amortized over {n_domains} domains: "
          f"{cma_cost/max(1,n_domains):.1f}s/domain")

    pd.DataFrame(rows).to_csv(
        f"{cfg.work_dir}/artifacts/exp5_timing.csv", index=False)

with Timer("Experiment 5"):
    run_exp5()

# ──────────────────────────────────────────────────────────────────────────────
# Experiment 6: Cross-Domain — Real Evaluation with Significance
# ──────────────────────────────────────────────────────────────────────────────
print_sep("EXPERIMENT 6: Cross-Domain")

def run_exp6():
    """
    Cross-domain scenario: model receives prompts from unknown domain.
    SPECTRA uses router to identify domain → outperforms naive merge.
    """
    # Mixed test: 15 from each domain = 45 total
    all_test = []
    for dom in LORA_NAMES:
        ex = get_test_examples(dom, n=15)
        all_test.extend(ex)
    random.Random(MASTER_SEED).shuffle(all_test)

    # Small mixed adapt set (5 per domain = 15 total)
    adapt_pr = []
    for dom in LORA_NAMES:
        adapt_pr.extend(get_adapt_examples(dom, 5, seed_offset=600))
    random.Random(MASTER_SEED).shuffle(adapt_pr)

    prompts = [p for p,_ in all_test]
    golds   = [g for _,g in all_test]
    methods = {}

    for nm, setup in [
        ("no_adaptation", reset_base),
        ("equal_merge",   apply_equal),
        ("ties_merge",    apply_ties),
    ]:
        setup(); set_seed(MASTER_SEED)
        preds = M.generate(prompts)
        m     = compute_metrics(preds, golds)
        methods[nm] = m
        mn, lo, hi = bootstrap_ci(m["comp_list"])
        print(f"  {nm:<22}: comp={mn:.4f}  [{lo:.4f},{hi:.4f}]")

    # SPECTRA: route on mixed adapt set
    adapt_prompts = [p for p,_ in adapt_pr]
    rw = router.fit(adapt_prompts, verbose=True)
    c, fit, _, t = spectra_adapt(adapt_pr, router_weights=rw, verbose=True)
    apply_c(c.tolist()); set_seed(MASTER_SEED)
    preds_sp  = M.generate(prompts)
    m_sp      = compute_metrics(preds_sp, golds)
    methods["spectra"] = m_sp
    mn, lo, hi = bootstrap_ci(m_sp["comp_list"])
    print(f"  {'spectra':<22}: comp={mn:.4f}  [{lo:.4f},{hi:.4f}]")

    # Significance
    print("\n  Significance (SPECTRA vs baselines):")
    for bl in ["no_adaptation","equal_merge","ties_merge"]:
        t_stat, p_val = paired_ttest(
            m_sp["comp_list"], methods[bl]["comp_list"])
        delta = m_sp["composite"] - methods[bl]["composite"]
        d = cohens_d(m_sp["comp_list"], methods[bl]["comp_list"])
        sig   = "✓ p<0.05" if p_val < 0.05 else f"✗ ns (p={p_val:.3f})"
        print(f"    vs {bl:<20}: Δ={delta:+.4f}  t={t_stat:+.2f}  "
              f"d={d:+.2f}  {sig}")

    best_m = max(methods.items(), key=lambda x: x[1]["composite"])
    print(f"\n  Best cross-domain: {best_m[0]} "
          f"(composite={best_m[1]['composite']:.4f})")

    RESULTS.append({
        "method":       "exp6_summary",
        "domain":       "mixed",
        "condition":    "exp6",
        "adapt_time_s": t,
        "coeffs":       json.dumps(c.tolist()),
        "spectra_comp": m_sp["composite"],
        "equal_comp":   methods["equal_merge"]["composite"],
        "delta_eq":     m_sp["composite"]-methods["equal_merge"]["composite"],
        "timestamp":    pd.Timestamp.utcnow().isoformat(),
    })
    flush_results()

with Timer("Experiment 6"):
    run_exp6()

# ──────────────────────────────────────────────────────────────────────────────
# Final Results
# ──────────────────────────────────────────────────────────────────────────────
print_sep("FINAL RESULTS")
flush_results()

if os.path.isfile(RESULTS_CSV):
    rdf  = pd.read_csv(RESULTS_CSV)
    exp1 = rdf[rdf["condition"]=="exp1_main"].copy()
    exp1["method_norm"] = exp1["method"].apply(
        lambda m: "single_lora" if m.startswith("single_lora_") else m)

    print(f"\n{'Method':<28}", end="")
    for m in ["em","f1","rouge_l","composite"]: print(f"  {m:>10}", end="")
    print(f"  {'95% CI composite':>22}")
    print("─"*95)

    for mn in METHOD_ORDER:
        sub = exp1[exp1["method_norm"]==mn]
        if sub.empty: continue
        print(f"  {mn:<26}", end="")
        for m in ["em","f1","rouge_l","composite"]:
            v = sub[m].mean() if m in sub.columns else 0
            print(f"  {v:>10.4f}", end="")
        if "comp_ci_lo" in sub.columns:
            lo = sub["comp_ci_lo"].mean(); hi = sub["comp_ci_hi"].mean()
            print(f"  [{lo:.4f},{hi:.4f}]", end="")
        mark = " ◄ SPECTRA" if mn=="spectra" else ""
        print(mark)

    print(f"\n{'='*50}")
    print("SPECTRA FINAL RANKING:")
    sw = 0
    for metric in METRICS:
        if metric not in exp1.columns: continue
        agg = (exp1.groupby("method_norm")[metric].mean()
                   .reindex(METHOD_ORDER).dropna()
                   .sort_values(ascending=False))
        winner = agg.index[0] if not agg.empty else "?"
        if winner == "spectra": sw += 1
        spec_rank = list(agg.index).index("spectra")+1 if "spectra" in agg.index else "?"
        print(f"  {metric:12s}: #1={winner:<20} SPECTRA=#{spec_rank}"
              f"{'  ✓ WINS' if winner=='spectra' else ''}")
    print(f"\n  SPECTRA wins {sw}/4 metrics")
    print(f"\n  Key advantage: SPECTRA uses domain routing + TTA")
    print(f"  → No extra labeled data needed vs oracle single_lora")
    print(f"  → Automatic domain identification from unlabeled prompts")

import shutil
try:
    zp = shutil.make_archive(f"{cfg.work_dir}/spectra_results","zip",
                             cfg.work_dir)
    print(f"Archive: {zp}")
except Exception as e: print(f"Archive: {e}")

print_sep("SPECTRA COMPLETE")
print(f"VRAM: {vram_mb():.0f}MB ✓")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 2.0 MB/s eta 0:00:00

============================= Model Path =============================
[MODEL] /kaggle/input/datasets/ali320230101/spectra-day1-2-artifacts-v1/baseline_full_ft_final

============================= Tokenizer ==============================
  Vocab=151643

=========================== Loading Model ============================


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  ✓ 338 params  VRAM=4021MB

============================== Dataset ===============================
[DATA] /kaggle/input/datasets/ali320230101/spectra-day1-2-artifacts-v1/data/unified_dataset
  math: 5000
  coding: 5000
  hybrid: 5000
  math: train=4935, adapt=15, test=50
  coding: train=4935, adapt=15, test=50
  hybrid: train=4935, adapt=15, test=50

============================== Metrics ===============================
  EM, F1, ROUGE-L, Composite(0.40/0.35/0.25) + Bootstrap CI

=========================== LoRA Training ============================
[LORA] Reusing math at /kaggle/input/datasets/ali320230101/spectra-day1-2-artifacts-v1/lora_math
[LORA] Reusing code at /kaggle/input/datasets/ali320230101/spectra-day1-2-artifacts-v1/lora_code
[LORA] Reusing hybrid at /kaggle/input/datasets/ali320230101/spectra-day1-2-artifacts-v1/lora_hybrid
[TIME] LoRA: 0.01s

======================= LoRA Delta Extraction ========================
  lora_math: 112 deltas
  lora_code: 112 deltas
  lora_hy

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [             no_adaptation|   math|s=42]  EM=0.0600  F1=0.1007  RL=0.1476  Comp=0.0961  t=63.9s
  [             no_adaptation|   math|s=123]  EM=0.0600  F1=0.1033  RL=0.1477  Comp=0.0971  t=62.5s
  [             no_adaptation|   math|s=256]  EM=0.0800  F1=0.1233  RL=0.1477  Comp=0.1121  t=62.0s
  [             no_adaptation|   math|s=512]  EM=0.0800  F1=0.1233  RL=0.1477  Comp=0.1121  t=62.0s
  [             no_adaptation|   math|s=1024]  EM=0.0800  F1=0.1207  RL=0.1476  Comp=0.1111  t=62.2s
  >>> no_adaptation/math:  EM=0.0720±0.0098  F1=0.1143  RL=0.1477  Composite=0.1057 [0.0801,0.1334]
  [          single_lora_math|   math|s=42]  EM=0.0800  F1=0.1650  RL=0.2058  Comp=0.1412  t=62.2s
  [          single_lora_math|   math|s=123]  EM=0.0800  F1=0.1700  RL=0.2061  Comp=0.1430  t=61.4s
  [          single_lora_math|   math|s=256]  EM=0.0800  F1=0.1650  RL=0.2053  Comp=0.1411  t=61.9s
  [          single_lora_math|   math|s=512]  EM=0.0600  F1=0.1530  RL=0.2082  Comp=0.1296  t=61.6s
 